# Docling End-to-End Document Processing

## Overview
This notebook demonstrates the use of the Docling library to process various document formats, including PDF, PPTX, and images. It covers:
- Simple PDF to Markdown conversion
- Multi-format document conversion
- Figure and table extraction
- Full-page OCR for scanned documents
- Translation of extracted text to German

## Prerequisites
- Input files: `2408.09869v5.pdf`, `demo.docx`, `file-7mNhEGHpd0.png`
- Tesseract installed for OCR
- Internet access for model downloads and Google Translate

## Instructions
Run the cells in sequence. Ensure input files are in the `/content/` directory or update paths accordingly.

This section installs all necessary libraries for document parsing, OCR, translation, and semantic search using the Docling library and its dependencies. Run this before executing the rest of the notebook.

In [ ]:
!pip install docling==2.102.1
!pip install rapidocr onnxruntime tesserocr pdf2image docling-core pandas easyocr
!pip install sentence-transformers langchain-community langchain-huggingface faiss-cpu
!pip install --upgrade huggingface_hub httpx
!pip install googletrans==4.0.2
!pip install transformers==4.55.4

  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Using cached huggingface_hub-1.23.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-1.23.0-py3-none-any.whl (770 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into ac

In [ ]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "docling",
    "docling-core",
    "rapidocr",
    "onnxruntime",
    "tesserocr",
    "pdf2image",
    "pandas",
    "easyocr",
    "sentence-transformers",
    "langchain-community",
    "langchain-huggingface",
    "faiss-cpu",
    "huggingface_hub",
    "httpx",
    "googletrans",
    "transformers",
    "torch",
]

print("=" * 60)
print("Installed Package Versions")
print("=" * 60)

for pkg in packages:
    try:
        print(f"{pkg:<30} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<30} Not Installed")

Installed Package Versions
docling                        2.102.1
docling-core                   2.87.1
rapidocr                       3.9.1
onnxruntime                    1.27.0
tesserocr                      2.10.0
pdf2image                      1.17.0
pandas                         2.2.2
easyocr                        1.7.2
sentence-transformers          5.6.0
langchain-community            0.4.2
langchain-huggingface          1.2.2
faiss-cpu                      1.14.3
huggingface_hub                0.36.2
httpx                          0.28.1
googletrans                    4.0.2
transformers                   4.55.4
torch                          2.11.0+cpu


Imports every library and modules required for document conversion, OCR configuration, and pipeline execution using the Docling framework. It also initializes the Google Translate client for multilingual support.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from docling.document_converter import DocumentConverter, PdfFormatOption, WordFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TesseractCliOcrOptions
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption
from docling.datamodel.pipeline_options import PictureDescriptionVlmOptions
from docling.pipeline.simple_pipeline import SimplePipeline
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling_core.types.doc import ImageRefMode, TableItem, TextItem

## 📄 Converting a PDF to Markdown using Docling

In this example, we use **Docling** to convert a PDF document into **Markdown**. During the conversion process, Docling analyzes the document layout and extracts structured content such as headings, paragraphs, tables, lists, and image captions.

The extracted content is then exported as a Markdown file for further use.

### 🔍 What this code does

- Loads the PDF document (`2408.09869v5.pdf`).
- Converts the document using `DocumentConverter`.
- Exports the parsed document as Markdown.
- Saves the Markdown output to a local `.md` file.

---

### 🔄 Conversion Workflow

```text
PDF Document
      │
      ▼
DocumentConverter
      │
      ▼
Structured Document
      │
      ▼
Markdown Export
      │
      ▼
Saved as .md File
```

---

### 📌 Why export to Markdown?

Markdown is a lightweight and structured format that is easy to read and process. It is commonly used for:

- Retrieval-Augmented Generation (RAG)
- Semantic Search
- Question Answering
- Document Summarization
- Knowledge Base Creation

In [ ]:
import os

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# Configure pipeline
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

source = "/content/2408.09869v5.pdf"

result = converter.convert(source)

output_filename = os.path.splitext(os.path.basename(source))[0] + ".md"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(result.document.export_to_markdown())

print(f"Markdown saved to: {output_filename}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/2408.09869v5.pdf'

## 📄 Converting a DOCX Document to Markdown using Docling

In this example, we use **Docling** to convert a Microsoft Word (**.docx**) document into a **Markdown (.md)** file. Docling preserves the document's structure while generating a clean, lightweight Markdown representation.

### 🔍 What this code does

- Initializes a `DocumentConverter` with the default configuration.
- Loads the DOCX document (`demo.docx`).
- Converts the document into Docling's structured document format.
- Exports the document as Markdown.
- Saves the generated Markdown using the same filename as the original document.

---

### 🔄 Conversion Workflow

```text
DOCX Document
      │
      ▼
DocumentConverter
      │
      ▼
Structured Docling Document
      │
      ▼
Markdown Export
      │
      ▼
Saved as .md File
```

---

### 🧩 Components Used

#### **DocumentConverter**
The main Docling component responsible for reading the input document and converting it into a structured internal representation.

#### **export_to_markdown()**
Exports the structured document as a Markdown file while preserving elements such as:
- Headings
- Paragraphs
- Lists
- Tables
- Basic document formatting

---

### 📌 Output

The resulting Markdown file contains the content and structure of the original DOCX document in a lightweight, human-readable format. This output can be used for:

- Documentation
- Version Control (Git)
- Retrieval-Augmented Generation (RAG)
- Semantic Search
- Knowledge Base Creation
- Further NLP Processing

In [ ]:
source = "/content/demo.docx"

converter = DocumentConverter()
result = converter.convert(source)


# Extract the filename from the source URL
filename = os.path.basename(source)

# Remove the file extension and add .md
output_filename = os.path.splitext(filename)[0] + ".md"

with open(output_filename, "w") as f:
    f.write(result.document.export_to_markdown())

## 🖼️ Converting a PDF with Custom Pipeline Options

In this example, we customize Docling's PDF processing pipeline by enabling **picture extraction**. Instead of using the default conversion settings, we configure `PdfPipelineOptions` to generate images for figures detected in the document.

This allows the converted document to contain both the extracted content and the associated figure images.

### 🔍 What this code does

- Creates a custom `PdfPipelineOptions` configuration.
- Enables picture extraction using `generate_picture_images=True`.
- Configures `DocumentConverter` to use the custom pipeline.
- Converts the PDF document (`2408.09869v5.pdf`).
- Returns the converted document as a `DoclingDocument`.

---

### 🔄 Conversion Workflow

```text
PDF Document
      │
      ▼
PdfPipelineOptions
(generate_picture_images=True)
      │
      ▼
DocumentConverter
      │
      ▼
DoclingDocument
(with extracted figures)
```

---

### 📌 Why use `PdfPipelineOptions`?

`PdfPipelineOptions` allows you to customize how Docling processes PDF documents. Depending on your use case, you can enable or disable features such as:

- OCR
- Table extraction
- Figure extraction
- Image generation
- Layout analysis

This flexibility allows the conversion pipeline to be tailored for different document-processing and RAG workflows.

In [ ]:
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    InputFormat,
)
from docling.datamodel.pipeline_options import PdfPipelineOptions

# Configure the PDF pipeline
pipeline_options = PdfPipelineOptions(
    generate_picture_images=True
)

# Disable table extraction (avoids the hanging issue)
pipeline_options.do_table_structure = False

# Create a converter with custom pipeline options
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

# Convert the PDF
result = converter.convert("/content/2408.09869v5.pdf")

# Get the structured document
doc = result.document

print("✅ PDF converted successfully!")

In [ ]:
from IPython.display import Image, display

for item, _ in doc.iterate_items():
    if item.label == "picture":
        image_data = item.image

        # Get the image URI
        uri = str(image_data.uri)

        # Display the image using IPython
        display(Image(url=uri))
        break

## 🏗️ Exploring the Structure of a Docling Document

In this example, we use Docling to convert a PDF into a structured `DoclingDocument` and then inspect the different types of elements detected within the document.

Instead of working with plain text, Docling organizes the document into structured elements such as headings, paragraphs, tables, figures, lists, and captions. This makes it easier to analyze and process documents programmatically.

### 🔍 What this code does

- Converts the PDF document (`2408.09869v5.pdf`).
- Creates a `DoclingDocument`.
- Iterates through every element in the document.
- Groups elements based on their type (label).
- Displays the document structure.

---

### 🔄 Processing Workflow

```text
PDF Document
      │
      ▼
DocumentConverter
      │
      ▼
DoclingDocument
      │
      ▼
Iterate Through Elements
      │
      ▼
Group by Element Type
      │
      ▼
Document Structure Breakdown
```

---

### 📌 Why inspect document structure?

Unlike traditional PDF parsers that return only plain text, Docling preserves the structure of the document. By grouping elements based on their type, you can better understand the document's layout and selectively process specific content.

Some common element types include:

- Titles
- Headings
- Paragraphs
- Tables
- Figures
- Lists
- Captions

This structured representation is particularly useful for applications such as document analysis, information extraction, and Retrieval-Augmented Generation (RAG).

In [ ]:
from collections import defaultdict

from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    InputFormat,
)
from docling.datamodel.pipeline_options import PdfPipelineOptions

# Configure the PDF pipeline
pipeline_options = PdfPipelineOptions()

# Disable table structure extraction
pipeline_options.do_table_structure = False

# Initialize the converter
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

# Convert the PDF
source = "/content/2408.09869v5.pdf"
result = converter.convert(source)

# Access the structured document
doc = result.document

print(f"Successfully processed document: {source}")

# Group document elements by their type
element_types = defaultdict(list)

for item, _ in doc.iterate_items():
    element_types[item.label].append(item)

# Display the document structure
print("Document structure breakdown:")

for element_type, items in element_types.items():
    print(f"{element_type}: {len(items)}")

##Print first section header

In [ ]:
# Print the first section heading

first_heading = element_types["section_header"][0]
print(first_heading.text)

##print the first element List data

In [ ]:
first_list_items = element_types["list_item"][0:6]
for list_item in first_list_items:
    print(list_item.text)

NameError: name 'element_types' is not defined

# 🤖 Building a Retrieval-Augmented Generation (RAG) Pipeline with Docling

In this example, we build a simple **Retrieval-Augmented Generation (RAG)** pipeline using Docling. The document is first converted into a structured format, then intelligently divided into chunks, embedded into a vector database, and finally queried using semantic similarity search.

This workflow demonstrates how Docling integrates with modern RAG pipelines by providing high-quality document parsing and intelligent chunking for downstream retrieval tasks.

### 🔄 RAG Workflow

```text
PDF Document
      │
      ▼
DocumentConverter
      │
      ▼
DoclingDocument
      │
      ▼
HybridChunker
      │
      ▼
Document Chunks
      │
      ▼
Embeddings
      │
      ▼
FAISS Vector Store
      │
      ▼
Similarity Search
      │
      ▼
Relevant Context
```

### 📌 Why use Docling for RAG?

Docling preserves the structure of documents before chunking, allowing the generated chunks to retain meaningful context. These structured chunks improve retrieval quality and provide better context for downstream language models during question answering.

### 📄 Converting the PDF

The first step is to convert the PDF into a structured **`DoclingDocument`** using `DocumentConverter`. Rather than extracting only plain text, Docling creates a rich, structured representation of the document that preserves its logical layout and organization. This `DoclingDocument` serves as the foundation for downstream tasks such as chunking, Retrieval-Augmented Generation (RAG), translation, metadata extraction, and exporting to formats like Markdown or HTML.

Instead of storing only text, the `DoclingDocument` organizes the document into structured objects such as:

- Titles
- Headings
- Paragraphs
- Lists
- Tables
- Pictures
- Captions
- Page information
- Metadata

This structured representation enables more intelligent document processing while preserving the relationships between different document elements.

In [ ]:
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    InputFormat,
)
from docling.datamodel.pipeline_options import PdfPipelineOptions

# Configure the PDF pipeline
pipeline_options = PdfPipelineOptions()

# Disable table structure extraction to improve processing time
pipeline_options.do_table_structure = False

# Initialize the converter
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

# Convert the PDF into a structured DoclingDocument
source = "/content/2408.09869v5.pdf"
result = converter.convert(source)

# Access the structured document
doc = result.document

print("✅ PDF successfully converted into a DoclingDocument.")

[INFO] 2026-07-14 11:45:52,937 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 11:45:52,972 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 11:45:52,973 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 11:45:53,036 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 11:45:53,041 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 11:45:53,042 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 11:45:53,092 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 11:45:53,171 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

✅ PDF successfully converted into a DoclingDocument.


### 📑 Helper Function for Viewing Chunks

This helper function displays the size and a preview of each generated chunk, making it easier to inspect the output produced by the chunking process.

In [ ]:
def print_chunk(chunk):
    print(f"Chunk length: {len(chunk.text)} characters")

    if len(chunk.text) > 30:
        print(
            f"Chunk content: "
            f"{chunk.text[:30]}..."
            f"{chunk.text[-30:]}"
        )
    else:
        print(f"Chunk content: {chunk.text}")

    print("-" * 50)

### ✂️ Chunking the Document

In this step, the structured document is divided into smaller, meaningful chunks using Docling's **HybridChunker**. The chunker considers the document structure while respecting the tokenizer's maximum token limit, producing chunks that are well suited for embedding and retrieval.

In [ ]:
from transformers import AutoTokenizer

from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import (
    HuggingFaceTokenizer,
)

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
    max_tokens=512,
)

chunker = HybridChunker(
    tokenizer=tokenizer
)

# Create intelligent chunks
rag_chunks = list(chunker.chunk(dl_doc=doc))

print(f"Created {len(rag_chunks)} intelligent chunks")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (702 > 512). Running this sequence through the model will result in indexing errors


Created 28 intelligent chunks


### 🧠 Creating the Vector Store

Each chunk is converted into a vector embedding using a sentence transformer model. These embeddings are then stored in a FAISS vector database, enabling efficient semantic search over the document.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Create the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Build the vector store
texts = [chunk.text for chunk in rag_chunks]

vectorstore = FAISS.from_texts(
    texts,
    embeddings
)

print(f"Built vector store with {len(texts)} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Built vector store with 28 chunks


### 🔍 Performing Semantic Search

Finally, the vector store is queried using a natural language question. FAISS retrieves the most semantically relevant document chunks, providing contextual information that can be supplied to a Large Language Model for Retrieval-Augmented Generation (RAG).

In [ ]:
# Search the knowledge base
query = "What is Docling? how does it work?"

relevant_docs = vectorstore.similarity_search(
    query,
    k=3
)

print(f"Query: {query}")
print(f"Found {len(relevant_docs)} relevant chunks:")

for i, doc in enumerate(relevant_docs, start=1):
    print(f"\nResult {i}:")
    print(doc.page_content[:150] + "...")

Query: What is Docling? how does it work?
Found 3 relevant chunks:

Result 1:
Docling implements a linear pipeline of operations, which execute sequentially on each given document (see Fig. 1). Each document is first parsed by a...

Result 2:
Docling is designed to allow easy extension of the model library and pipelines. In the future, we plan to extend Docling with several more models, suc...

Result 3:
In the final pipeline stage, Docling assembles all prediction results produced on each page into a well-defined datatype that encapsulates a converted...


## ⏱️ Measuring Document Conversion Time

In this example, we measure the time taken by Docling to convert a PDF document into Markdown. This helps evaluate the performance of the document conversion process while still generating the parsed output.

### 🔍 What this code does

- Loads the PDF document (`2408.09869v5.pdf`).
- Initializes the `DocumentConverter`.
- Records the start time before conversion.
- Converts the document into a `DoclingDocument`.
- Exports the document as Markdown.
- Calculates and displays the total conversion time.

---

### 🔄 Conversion Workflow

```text
PDF Document
      │
      ▼
Start Timer
      │
      ▼
DocumentConverter
      │
      ▼
DoclingDocument
      │
      ▼
Markdown Export
      │
      ▼
Stop Timer
      │
      ▼
Display Conversion Time
```

---

### 📌 Why measure conversion time?

Measuring the conversion time helps evaluate the performance of the document processing pipeline. This is particularly useful when working with:

- Large PDF documents
- Batch document processing
- Performance benchmarking
- Comparing different pipeline configurations
- Optimizing document-processing workflows

In [ ]:
import os
import time

from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    InputFormat,
)
from docling.datamodel.pipeline_options import PdfPipelineOptions

# Input PDF
source = "/content/2408.09869v5.pdf"

# Configure the PDF pipeline
pipeline_options = PdfPipelineOptions()

# Disable table structure extraction
pipeline_options.do_table_structure = False

# Initialize the converter
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

# Measure conversion time
start_time = time.time()
result = converter.convert(source)
end_time = time.time()

# Create the output Markdown filename
filename = os.path.basename(source)
output_filename = os.path.splitext(filename)[0] + ".md"

# Export the document as Markdown
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(result.document.export_to_markdown())

# Display the total conversion time
time_taken = end_time - start_time
print(f"Markdown saved to: {output_filename}")
print(f"Time taken to convert the document: {time_taken:.2f} seconds")

[INFO] 2026-07-14 11:48:06,873 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 11:48:06,908 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 11:48:06,909 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 11:48:06,971 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 11:48:06,976 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 11:48:06,977 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 11:48:07,025 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 11:48:07,099 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Markdown saved to: 2408.09869v5.md
Time taken to convert the document: 56.30 seconds


# ⚖️ Comparing Different Docling Parsing Pipelines

Docling supports multiple parsing backends and OCR engines, allowing the document conversion pipeline to be customized for different types of PDFs.

In the following examples, we compare several configurations, including PyPdfium, the default Docling parser, EasyOCR, Tesseract CLI, and RapidOCR. Each configuration processes the same document (`2408.09869v5.pdf`) so that their performance and output can be compared.

In [ ]:
import json
import logging
import time
from pathlib import Path
import os

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    RapidOcrOptions,
    TesseractOcrOptions,
    TesseractCliOcrOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.document_converter import DocumentConverter, PdfFormatOption
import os

# from docling.datamodel.pipeline_options import OcrMacOptions   # Uncomment if using macOS OCR
# os.environ["TESSDATA_PREFIX"] = r"C:\Program Files\Tesseract-OCR\"

_log = logging.getLogger(__name__)

# 📝 Helper function to save outputs in separate folder
def export_results(conv_result, folder_name):
    output_dir = Path(folder_name)
    output_dir.mkdir(parents=True, exist_ok=True)
    doc_filename = conv_result.input.file.stem

    with (output_dir / f"{doc_filename}.json").open("w", encoding="utf-8") as fp:
        fp.write(json.dumps(conv_result.document.export_to_dict()))

    with (output_dir / f"{doc_filename}.txt").open("w", encoding="utf-8") as fp:
        fp.write(conv_result.document.export_to_markdown())

    with (output_dir / f"{doc_filename}.md").open("w", encoding="utf-8") as fp:
        fp.write(conv_result.document.export_to_markdown())

    with (output_dir / f"{doc_filename}.doctags").open("w", encoding="utf-8") as fp:
        fp.write(conv_result.document.export_to_doctags())

## 📄 PyPdfium Backend without OCR

In this example, Docling uses the **PyPdfium** backend to process the PDF without performing Optical Character Recognition (OCR). This approach is suitable for digital PDFs that already contain selectable text, resulting in faster document conversion.

In [ ]:
# =============================================================
# PyPdfium without OCR
# =============================================================

logging.basicConfig(level=logging.INFO)

# Input PDF
input_doc_path = Path("/content/2408.09869v5.pdf")

# Configure the PDF pipeline
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False

# Disable table structure extraction
pipeline_options.do_table_structure = False

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            backend=PyPdfiumDocumentBackend,
        )
    }
)

# Convert the document
start = time.time()
conv_result = doc_converter.convert(input_doc_path)

print(f"✅ PyPdfium without OCR completed in {time.time() - start:.2f} seconds")

# Export results
export_results(
    conv_result,
    "output/PyPdfium_without_OCR",
)

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

✅ PyPdfium without OCR completed in 32.50 seconds


## 📄 PyPdfium Backend with EasyOCR

In this example, EasyOCR is enabled along with the **PyPdfium** backend. OCR is applied to detect and extract text from scanned or image-based pages while preserving the document structure.

In [ ]:
# =============================================================
# PyPdfium with EasyOCR
# =============================================================

from docling.datamodel.pipeline_options import EasyOcrOptions

pipeline_options = PdfPipelineOptions()

# Enable EasyOCR
pipeline_options.do_ocr = True
pipeline_options.ocr_options = EasyOcrOptions()

# Disable table structure extraction
pipeline_options.do_table_structure = False

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            backend=PyPdfiumDocumentBackend,
        )
    }
)

start = time.time()

conv_result = doc_converter.convert(input_doc_path)

print(f"✅ PyPdfium with EasyOCR completed in {time.time() - start:.2f} seconds")

export_results(
    conv_result,
    "output/PyPdfium_with_EasyOCR",
)

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume

✅ PyPdfium with EasyOCR completed in 137.63 seconds


## 📄 Docling Parser without OCR

This example uses Docling's default parsing pipeline without OCR. It extracts the document structure directly from the PDF, making it ideal for machine-generated documents that already contain embedded text.

In [ ]:
# =============================================================
# Docling Parse without OCR
# =============================================================

pipeline_options = PdfPipelineOptions()

# Disable OCR
pipeline_options.do_ocr = False

# Disable table structure extraction
pipeline_options.do_table_structure = False

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

start = time.time()

conv_result = doc_converter.convert(input_doc_path)

print(f"✅ Docling Parse without OCR completed in {time.time() - start:.2f} seconds")

export_results(
    conv_result,
    "output/Docling_without_OCR",
)

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

✅ Docling Parse without OCR completed in 35.51 seconds


## 📄 Docling Parser with EasyOCR

In this example, Docling's default parsing pipeline is combined with **EasyOCR**. OCR is automatically applied when needed, enabling the extraction of text from scanned or image-based documents while preserving the document layout.

In [ ]:
# =============================================================
# Docling Parse with EasyOCR (default)
# =============================================================

from docling.datamodel.pipeline_options import EasyOcrOptions

pipeline_options = PdfPipelineOptions()

# Enable EasyOCR
pipeline_options.do_ocr = True
pipeline_options.ocr_options = EasyOcrOptions()

# Disable table structure extraction
pipeline_options.do_table_structure = False

# Accelerator settings
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=4,
    device=AcceleratorDevice.AUTO,
)

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

start = time.time()

conv_result = doc_converter.convert(input_doc_path)

print(
    f"✅ Docling Parse with EasyOCR (default) completed in "
    f"{time.time() - start:.2f} seconds"
)

export_results(
    conv_result,
    "output/Docling_EasyOCR_Default",
)

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✅ Docling Parse with EasyOCR (default) completed in 118.39 seconds


## 📄 Docling Parser with EasyOCR (CPU)

This example demonstrates the same Docling pipeline using **EasyOCR** on the CPU. It is useful for systems without GPU acceleration while providing the same OCR capabilities.

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

CUDA Available: False
Running on CPU


In [ ]:
# =============================================================
# Docling Parse with EasyOCR (CPU only)
# =============================================================

from docling.datamodel.pipeline_options import EasyOcrOptions

pipeline_options = PdfPipelineOptions()

# Enable EasyOCR
pipeline_options.do_ocr = True
pipeline_options.ocr_options = EasyOcrOptions()

# Force EasyOCR to run on CPU
pipeline_options.ocr_options.use_gpu = False

# Disable table structure extraction
pipeline_options.do_table_structure = False

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

start = time.time()

conv_result = doc_converter.convert(input_doc_path)

print(
    f"✅ Docling Parse with EasyOCR (CPU only) completed in "
    f"{time.time() - start:.2f} seconds"
)

export_results(
    conv_result,
    "output/Docling_EasyOCR_CPU",
)

/usr/local/lib/python3.12/dist-packages/docling/models/stages/ocr/easyocr_model.py:70: UserWarning: Deprecated field. Better to set the `accelerator_options.device` in `pipeline_options`. When `use_gpu and accelerator_options.device == AcceleratorDevice.CUDA` the GPU is used to run EasyOCR. Otherwise, EasyOCR runs in CPU.
  warnings.warn(


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✅ Docling Parse with EasyOCR (CPU only) completed in 119.63 seconds


## 📄 Docling Parser with Tesseract CLI

In this example, Docling uses the **Tesseract CLI** as the OCR engine instead of EasyOCR. This allows you to compare OCR quality and performance using a widely adopted open-source OCR solution.

In [ ]:
# =============================================================
# Docling Parse with Tesseract CLI
# =============================================================

pipeline_options = PdfPipelineOptions()

# Enable Tesseract CLI OCR
pipeline_options.do_ocr = True
pipeline_options.ocr_options = TesseractCliOcrOptions()

# Disable table structure extraction
pipeline_options.do_table_structure = False

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

start = time.time()

conv_result = doc_converter.convert(input_doc_path)

print(
    f"✅ Docling Parse with Tesseract CLI completed in "
    f"{time.time() - start:.2f} seconds"
)

export_results(
    conv_result,
    "output/Tesseract_CLI",
)

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

ERROR:docling.models.stages.ocr.tesseract_ocr_cli_model:OSD failed (doc /content/2408.09869v5.pdf, page: 0, OCR rectangle: 0, processed image file <tempfile._TemporaryFileWrapper object at 0x7eb053b6e540>):
 b'Warning: Invalid resolution 0 dpi. Using 70 instead.\nEstimating resolution as 104\nWarning. Invalid resolution 0 dpi. Using 70 instead.\nToo few characters. Skipping this page\nError during processing.\n'


✅ Docling Parse with Tesseract CLI completed in 46.41 seconds


## 📄 Docling Parser with RapidOCR

This example configures Docling to use **RapidOCR** for text extraction. RapidOCR is a lightweight OCR engine designed for fast document processing while maintaining good text recognition accuracy.

In [ ]:
# =============================================================
# Docling Parse with RapidOCR
# =============================================================

pipeline_options = PdfPipelineOptions()

# Enable RapidOCR
pipeline_options.do_ocr = True
pipeline_options.ocr_options = RapidOcrOptions()

# Disable table structure extraction
pipeline_options.do_table_structure = False

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

start = time.time()

conv_result = doc_converter.convert(input_doc_path)

print(
    f"✅ Docling Parse with RapidOCR completed in "
    f"{time.time() - start:.2f} seconds"
)

export_results(
    conv_result,
    "output/RapidOCR",
)

[INFO] 2026-07-14 11:58:24,095 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 11:58:24,157 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 11:58:24,159 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 11:58:24,289 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 11:58:24,299 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 11:58:24,302 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 11:58:24,470 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 11:58:24,677 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

✅ Docling Parse with RapidOCR completed in 61.28 seconds


## 📂 Converting Multiple Documents using Docling

In this example, we use Docling to convert **multiple documents** in a single run. The converter is configured to support different document formats, allowing PDFs and images to be processed together.

### 🔍 What this code does

- Creates a list of input documents (PDF and PNG).
- Configures `DocumentConverter` with support for multiple file formats.
- Converts all documents using `convert_all()`.
- Exports each converted document as a separate Markdown file.

---

### 🔄 Conversion Workflow

```text
Multiple Documents
(PDF, Image, ...)
        │
        ▼
DocumentConverter
        │
        ▼
Document Processing
        │
        ▼
DoclingDocument Objects
        │
        ▼
Markdown Export
```

---

### 📌 Why use `convert_all()`?

Instead of converting documents one at a time, `convert_all()` processes multiple files in a single operation. This is useful for:

- Batch document processing
- Mixed document collections
- Large-scale document conversion
- Building document repositories for RAG

In [ ]:
import os
os.mkdir("Multi_scratch")

FileExistsError: [Errno 17] File exists: 'Multi_scratch'

In [ ]:
import logging
from pathlib import Path

from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TesseractCliOcrOptions,
)
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    WordFormatOption,
    ImageFormatOption,
)
from docling.pipeline.simple_pipeline import SimplePipeline
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline

_log = logging.getLogger(__name__)

# Input documents
input_doc_paths = [
    Path("/content/2408.09869v5.pdf"),
    Path("/content/file-7mNhEGHpd0.png"),
]

# Configure the PDF pipeline
pdf_pipeline_options = PdfPipelineOptions()

# Disable table structure extraction
pdf_pipeline_options.do_table_structure = False

# Initialize the converter with support for multiple formats
doc_converter = DocumentConverter(
    allowed_formats=[
        InputFormat.PDF,
        InputFormat.IMAGE,
        InputFormat.PPTX,
        InputFormat.DOCX,
        InputFormat.HTML,
        InputFormat.ASCIIDOC,
        InputFormat.CSV,
        InputFormat.MD,
    ],
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=StandardPdfPipeline,
            backend=PyPdfiumDocumentBackend,
            pipeline_options=pdf_pipeline_options,
        ),
        InputFormat.PPTX: WordFormatOption(
            pipeline_cls=SimplePipeline,
        ),
        InputFormat.IMAGE: ImageFormatOption(
            ocr_options=TesseractCliOcrOptions(),
        ),
    },
)

# Convert all documents
conv_results = doc_converter.convert_all(input_doc_paths)

# Output directory
out_path = Path("Multi_scratch")
out_path.mkdir(parents=True, exist_ok=True)

# Export each document as Markdown
for res in conv_results:
    print(
        f"Document {res.input.file.name} converted."
        f"\nSaved markdown output to: {out_path}"
    )

    with (out_path / f"{res.input.file.stem}.md").open(
        "w", encoding="utf-8"
    ) as fp:
        fp.write(res.document.export_to_markdown())

[INFO] 2026-07-14 13:30:59,999 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 13:31:00,176 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 13:31:00,180 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 13:31:00,643 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 13:31:00,662 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 13:31:00,664 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 13:31:00,776 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 13:31:01,011 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Document 2408.09869v5.pdf converted.
Saved markdown output to: Multi_scratch


[INFO] 2026-07-14 13:32:11,057 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 13:32:11,059 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 13:32:11,130 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 13:32:11,136 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 13:32:11,137 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 13:32:11,216 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 13:32:11,327 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_rec_small.onnx
[INFO] 2026-07-14

Document file-7mNhEGHpd0.png converted.
Saved markdown output to: Multi_scratch


## 🖼️ Exporting Pages and Figures

In this example, we configure **Docling** to extract not only the document text but also its visual content. During conversion, Docling generates images for each page and detected figures, then exports the document as Markdown and HTML with embedded or referenced images.

> **Note:** Table structure extraction is disabled (`do_table_structure=False`) to improve processing performance for this document. As a result, table images are not exported in this example.

### 🔍 What this code does

- Converts the PDF document (`2408.09869v5.pdf`).
- Generates images for every page in the document.
- Extracts and saves detected figures as separate image files.
- Exports the document as Markdown with embedded and referenced images.
- Exports the document as HTML with referenced images.

---

### 🔄 Export Workflow

```text
PDF Document
      │
      ▼
DocumentConverter
      │
      ▼
DoclingDocument
      │
      ├────────► Page Images
      │
      ├────────► Figure Images
      │
      ├────────► Markdown Export
      │
      └────────► HTML Export
```

---

### 📌 Why export images?

Many PDF documents contain valuable visual information such as figures, charts, diagrams, and page layouts. Exporting these visual elements separately makes them available for applications such as:

- Retrieval-Augmented Generation (RAG)
- Vision-language pipelines
- Document analysis
- Knowledge base creation
- Accessibility and image processing

In [ ]:
import os
os.mkdir("Figure export_scratch")

In [ ]:
import logging
import shutil
import time
from pathlib import Path

from docling_core.types.doc import ImageRefMode, PictureItem

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)

_log = logging.getLogger(__name__)

IMAGE_RESOLUTION_SCALE = 2.0

logging.basicConfig(level=logging.INFO)

# Input PDF
input_doc_path = Path("/content/2408.09869v5.pdf")

# Output directory
output_dir = Path("figure_export_output")

# Remove existing output directory
if output_dir.exists():
    shutil.rmtree(output_dir)

# Configure the PDF pipeline
pipeline_options = PdfPipelineOptions()
pipeline_options.images_scale = IMAGE_RESOLUTION_SCALE
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True

# Disable table structure extraction
pipeline_options.do_table_structure = False

# Initialize the converter
doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

start_time = time.time()

# Convert the document
conv_res = doc_converter.convert(input_doc_path)

output_dir.mkdir(parents=True, exist_ok=True)
doc_filename = conv_res.input.file.stem

# Save page images
for page in conv_res.document.pages.values():
    with (output_dir / f"{doc_filename}-{page.page_no}.png").open("wb") as fp:
        page.image.pil_image.save(fp, format="PNG")

# Save extracted figure images
picture_counter = 0

for element, _ in conv_res.document.iterate_items():

    if isinstance(element, PictureItem):
        picture_counter += 1

        with (
            output_dir /
            f"{doc_filename}-picture-{picture_counter}.png"
        ).open("wb") as fp:

            element.get_image(conv_res.document).save(fp, "PNG")

# Export Markdown with embedded images
conv_res.document.save_as_markdown(
    output_dir / f"{doc_filename}-with-images.md",
    image_mode=ImageRefMode.EMBEDDED,
)

# Export Markdown with referenced images
conv_res.document.save_as_markdown(
    output_dir / f"{doc_filename}-with-image-refs.md",
    image_mode=ImageRefMode.REFERENCED,
)

# Export HTML with referenced images
conv_res.document.save_as_html(
    output_dir / f"{doc_filename}-with-image-refs.html",
    image_mode=ImageRefMode.REFERENCED,
)

end_time = time.time() - start_time

_log.info(
    f"Document converted and figures exported in {end_time:.2f} seconds."
)

[INFO] 2026-07-14 12:02:03,665 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 12:02:03,701 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 12:02:03,702 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 12:02:03,772 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 12:02:03,779 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 12:02:03,781 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 12:02:03,841 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 12:02:03,910 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

## 🖼️ Extracting Text from an Image using Full-Page OCR

In this example, Docling processes an image using **Tesseract OCR** with **full-page OCR** enabled. Instead of extracting text only from detected regions, the entire image is analyzed to recognize all visible text.

### 🔍 What this code does

- Loads the image (`file-7mNhEGHpd0.png`).
- Configures Tesseract OCR with full-page text recognition.
- Converts the image into a structured `DoclingDocument`.
- Exports the extracted content as a Markdown file.

---

### 🔄 OCR Workflow

```text
Image Document
       │
       ▼
Tesseract OCR
(Full-Page OCR)
       │
       ▼
DoclingDocument
       │
       ▼
Markdown Export
       │
       ▼
Saved as .md File
```

---

### 📌 Why use Full-Page OCR?

Full-page OCR analyzes the entire image instead of relying only on detected text regions. This is particularly useful for scanned documents, forms, receipts, screenshots, and other image-based documents where text may appear throughout the page.

In [ ]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import TesseractCliOcrOptions
from docling.document_converter import (
    DocumentConverter,
    ImageFormatOption,
)


def main():
    # Input image
    input_doc_path = Path("/content/file-7mNhEGHpd0.png")

    # Output directory
    output_dir = Path("full_figure_ocr")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Configure OCR
    ocr_options = TesseractCliOcrOptions(
        force_full_page_ocr=True
    )

    # Initialize the converter
    converter = DocumentConverter(
        format_options={
            InputFormat.IMAGE: ImageFormatOption(
                ocr_options=ocr_options
            )
        }
    )

    # Convert the image
    result = converter.convert(input_doc_path)
    doc = result.document

    # Export as Markdown
    md = doc.export_to_markdown()

    output_path = output_dir / f"{input_doc_path.stem}.md"
    output_path.write_text(md, encoding="utf-8")

    print(f"✅ Markdown saved at: {output_path.resolve()}")


if __name__ == "__main__":
    main()

[INFO] 2026-07-14 12:08:32,565 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 12:08:32,651 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 12:08:32,652 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 12:08:32,795 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 12:08:32,803 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 12:08:32,805 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 12:08:32,924 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 12:08:33,062 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

✅ Markdown saved at: /content/full_figure_ocr/file-7mNhEGHpd0.md


# 🌍 Translating a Document after Conversion

In this example, we use Docling to convert a PDF into a structured `DoclingDocument`, translate its textual content using **Google Translate**, and export both the original and translated versions as Markdown files.

### 🔍 What this code does

- Converts the PDF document (`2408.09869v5.pdf`).
- Saves the original document as Markdown.
- Translates text in paragraphs and table cells.
- Exports a translated Markdown version while preserving the document structure and images.

---

### 🔄 Translation Workflow

```text
PDF Document
      │
      ▼
DocumentConverter
      │
      ▼
DoclingDocument
      │
      ▼
Translate Text
(Google Translate)
      │
      ▼
Updated DoclingDocument
      │
      ▼
Translated Markdown Export
```

---

### 📌 Why translate a `DoclingDocument`?

Since Docling represents documents as structured objects, their contents can be modified programmatically before export. This makes it possible to build workflows such as:

- Document translation
- Content localization
- Multilingual knowledge bases
- International document processing
- Retrieval-Augmented Generation (RAG) across multiple languages

In [ ]:
import os
os.mkdir("Simpletranslation_scratch")

In [ ]:
import logging
import asyncio
from pathlib import Path
from collections import Counter

import nest_asyncio
from googletrans import Translator

from docling_core.types.doc import ImageRefMode

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)

# Allow asyncio inside Colab/Jupyter
nest_asyncio.apply()

translator = Translator()
_log = logging.getLogger(__name__)

IMAGE_RESOLUTION_SCALE = 2.0


async def translate(text: str, src: str = "en", dest: str = "de") -> str:
    """Translate text asynchronously using Google Translate."""

    if not text or not text.strip():
        return text

    try:
        translated = await translator.translate(
            text,
            src=src,
            dest=dest,
        )
        return translated.text

    except Exception as e:
        _log.warning(f"Translation failed: {e}")
        return text


def translate_sync(text: str, src: str = "en", dest: str = "de") -> str:
    """Synchronous wrapper around the async translator."""
    return asyncio.get_event_loop().run_until_complete(
        translate(text, src, dest)
    )


def main():
    logging.basicConfig(level=logging.INFO)

    # Input PDF
    input_doc_path = Path("/content/2408.09869v5.pdf")

    # Output directory
    output_dir = Path("translated_document_output")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Configure the PDF pipeline
    pipeline_options = PdfPipelineOptions()
    pipeline_options.images_scale = IMAGE_RESOLUTION_SCALE
    pipeline_options.generate_page_images = True
    pipeline_options.generate_picture_images = True

    # Disable table extraction
    pipeline_options.do_table_structure = False

    # Initialize converter
    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options
            )
        }
    )

    # Convert document
    conv_res = doc_converter.convert(input_doc_path)
    conv_doc = conv_res.document
    doc_filename = conv_res.input.file.stem

    # Save original Markdown
    conv_doc.save_as_markdown(
        output_dir / f"{doc_filename}-original.md",
        image_mode=ImageRefMode.EMBEDDED,
    )

    # Show element types
    counter = Counter()

    for element, _ in conv_doc.iterate_items():
        counter[type(element).__name__] += 1

    print("\nDocument element types:")
    for name, count in sorted(counter.items()):
        print(f"{name}: {count}")

    # Translate all elements that contain text
    translated_elements = 0

    for element, _ in conv_doc.iterate_items():

        if hasattr(element, "text") and isinstance(element.text, str):

            if element.text.strip():

                original = element.text
                translated = translate_sync(original)

                if translated != original:
                    element.text = translated
                    translated_elements += 1

    print(f"\nTranslated elements: {translated_elements}")

    # Save translated Markdown
    conv_doc.save_as_markdown(
        output_dir / f"{doc_filename}-translated.md",
        image_mode=ImageRefMode.EMBEDDED,
    )

    print(f"\nTranslated Markdown saved to: {output_dir}")


if __name__ == "__main__":
    main()

[INFO] 2026-07-14 12:08:59,385 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 12:08:59,448 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 12:08:59,449 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-14 12:08:59,544 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 12:08:59,552 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 12:08:59,554 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-14 12:08:59,643 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-14 12:08:59,750 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]


Document element types:
CodeItem: 1
ListItem: 23
PictureItem: 5
SectionHeaderItem: 22
TableItem: 3
TextItem: 271

Translated elements: 221

Translated Markdown saved to: translated_document_output
